#### Day 5
- Author cleaning plan (missing values, duplicates, type fixes, category harmonisation)
- Decide keep / drop / impute rule for each affected column
- Freeze cleaning plan v1 and commit to /docs

### Author Cleaning Plan

In [2]:
import pandas as pd
import numpy as np

# ============================================================
# TASK 1: AUTHOR DATA CLEANING PLAN - ISSUE IDENTIFICATION
# ============================================================

# Load dataset
df = pd.read_csv("project_dataset.csv")

print("=" * 90)
print("TASK 1: DATA CLEANING PLAN - ISSUE IDENTIFICATION")
print("=" * 90)

print("\nDataset Shape:")
print("Rows   :", df.shape[0])
print("Columns:", df.shape[1])


# ============================================================
# 1. MISSING VALUES
# ============================================================

missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isnull().sum().values,
    "Missing_Percentage": (
        df.isnull().mean().values * 100
    ).round(2)
})

missing_report = missing_report[
    missing_report["Missing_Count"] > 0
].sort_values(
    by="Missing_Count",
    ascending=False
)

print("\n" + "=" * 90)
print("1. MISSING VALUES")
print("=" * 90)

if not missing_report.empty:
    print(missing_report.to_string(index=False))
else:
    print("No missing values detected.")


# ============================================================
# 2. DUPLICATE CHECK
# ============================================================

print("\n" + "=" * 90)
print("2. DUPLICATE CHECK")
print("=" * 90)

# Complete duplicate rows
duplicate_rows = df.duplicated().sum()

print("Complete Duplicate Rows:", duplicate_rows)

# Duplicate movie_id
if "movie_id" in df.columns:

    duplicate_movie_ids = df.duplicated(
        subset=["movie_id"],
        keep=False
    )

    duplicate_movie_id_count = (
        df.loc[duplicate_movie_ids, "movie_id"]
        .duplicated()
        .sum()
    )

    print(
        "Duplicate movie_id Count:",
        duplicate_movie_id_count
    )

    if duplicate_movie_ids.any():

        print("\nDuplicate movie_id Records:")

        display(
            df.loc[
                duplicate_movie_ids,
                ["movie_id", "title"]
            ].sort_values("movie_id")
        )

    else:
        print("No duplicate movie_id values detected.")

else:
    print("movie_id column not found.")


# ============================================================
# 3. DATA TYPE CHECK
# ============================================================

print("\n" + "=" * 90)
print("3. DATA TYPES")
print("=" * 90)

datatype_report = pd.DataFrame({
    "Column": df.columns,
    "Detected_Data_Type": df.dtypes.astype(str).values
})

print(datatype_report.to_string(index=False))


# ============================================================
# 4. RELEASE DATE VALIDATION
# ============================================================

print("\n" + "=" * 90)
print("4. RELEASE DATE VALIDATION")
print("=" * 90)

if "Release_Date" in df.columns:

    # dayfirst=True handles dates such as:
    # 08-08-2006
    # 16-04-2003
    # 20-11-2015

    parsed_dates = pd.to_datetime(
        df["Release_Date"],
        errors="coerce",
        dayfirst=True
    )

    malformed_date_mask = (
        df["Release_Date"].notna()
        &
        parsed_dates.isna()
    )

    malformed_dates = malformed_date_mask.sum()

    print(
        "Malformed Release_Date Values:",
        malformed_dates
    )

    if malformed_dates > 0:

        print("\nMalformed Date Records:")

        columns_to_show = [
            col for col in
            ["movie_id", "title", "Release_Date"]
            if col in df.columns
        ]

        display(
            df.loc[
                malformed_date_mask,
                columns_to_show
            ]
        )

    else:
        print(
            "No malformed Release_Date values detected."
        )

else:
    print("Release_Date column not found.")


# ============================================================
# 5. RELEASE YEAR VALIDATION
# ============================================================

print("\n" + "=" * 90)
print("5. RELEASE YEAR VALIDATION")
print("=" * 90)

if "Release_Year" in df.columns:

    numeric_year = pd.to_numeric(
        df["Release_Year"],
        errors="coerce"
    )

    # Non-numeric years
    invalid_year_mask = (
        df["Release_Year"].notna()
        &
        numeric_year.isna()
    )

    invalid_years = invalid_year_mask.sum()

    print(
        "Non-Numeric Release_Year Values:",
        invalid_years
    )

    # Basic reasonable range validation
    out_of_range_year_mask = (
        numeric_year.notna()
        &
        (
            (numeric_year < 1888)
            |
            (numeric_year > pd.Timestamp.now().year)
        )
    )

    print(
        "Out-of-Range Release_Year Values:",
        out_of_range_year_mask.sum()
    )


    # Compare Release_Year with Release_Date
    if "Release_Date" in df.columns:

        extracted_year = parsed_dates.dt.year

        year_mismatch_mask = (
            extracted_year.notna()
            &
            numeric_year.notna()
            &
            (extracted_year != numeric_year)
        )

        print(
            "Release_Date vs Release_Year Mismatches:",
            year_mismatch_mask.sum()
        )

else:
    print("Release_Year column not found.")


# ============================================================
# 6. NUMERIC-AS-TEXT CHECK
# ============================================================

print("\n" + "=" * 90)
print("6. NUMERIC-AS-TEXT CHECK")
print("=" * 90)

# Columns expected to be numerical
expected_numeric_columns = [
    "Budget",
    "Revenue",
    "Profit",
    "ROI",
    "Revenue_Budget_Ratio",
    "Popularity",
    "Genre_Count",
    "Production_Company_Count",
    "Cast_Count",
    "Crew_Count",
    "Male_Cast_Count",
    "Female_Cast_Count",
    "Male_Crew_Count",
    "Female_Crew_Count",
    "Cast_Gender_Ratio",
    "Crew_Gender_Ratio",
    "Producer_Count",
    "Writer_Count",
    "Total_People_Involved",
    "Runtime",
    "Release_Year"
]

numeric_type_report = []

for col in expected_numeric_columns:

    if col in df.columns:

        converted = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        invalid_numeric = (
            df[col].notna()
            &
            converted.isna()
        ).sum()

        numeric_type_report.append({
            "Column": col,
            "Current_Type": str(df[col].dtype),
            "Invalid_Numeric_Values": invalid_numeric
        })

numeric_type_report = pd.DataFrame(
    numeric_type_report
)

if not numeric_type_report.empty:

    print(
        numeric_type_report.to_string(
            index=False
        )
    )


# ============================================================
# 7. IMPORTANT NUMERIC VALUE CHECK
# ============================================================

print("\n" + "=" * 90)
print("7. ZERO / NEGATIVE VALUE CHECK")
print("=" * 90)

important_numeric = [
    "Revenue",
    "Genre_Count",
    "Production_Company_Count"
]

numeric_issue_report = []

for col in important_numeric:

    if col in df.columns:

        values = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        numeric_issue_report.append({

            "Column":
                col,

            "Missing":
                values.isna().sum(),

            "Zero_Values":
                (values == 0).sum(),

            "Negative_Values":
                (values < 0).sum(),

            "Minimum":
                values.min(),

            "Maximum":
                values.max()
        })

numeric_issue_report = pd.DataFrame(
    numeric_issue_report
)

print(
    numeric_issue_report.to_string(
        index=False
    )
)


# ============================================================
# 8. WHITESPACE CHECK
# ============================================================

print("\n" + "=" * 90)
print("8. WHITESPACE CHECK")
print("=" * 90)

text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

whitespace_report = []

for col in text_columns:

    values = df[col].dropna().astype(str)

    whitespace_count = (
        values != values.str.strip()
    ).sum()

    if whitespace_count > 0:

        whitespace_report.append({

            "Column":
                col,

            "Whitespace_Issues":
                whitespace_count
        })

whitespace_report = pd.DataFrame(
    whitespace_report
)

if not whitespace_report.empty:

    print(
        whitespace_report.to_string(
            index=False
        )
    )

else:

    print(
        "No leading or trailing whitespace issues detected."
    )


# ============================================================
# 9. IMPORTANT CATEGORY REVIEW
# ============================================================

print("\n" + "=" * 90)
print("9. IMPORTANT CATEGORY REVIEW")
print("=" * 90)

category_columns = [
    "Production_Companies",
    "Primary_Production_Company",
    "Cast_Gender_Ratio"
]

for col in category_columns:

    if col in df.columns:

        print("\n" + "-" * 70)
        print(f"TOP 20 VALUES: {col}")
        print("-" * 70)

        print(
            df[col]
            .value_counts(
                dropna=False
            )
            .head(20)
        )


# ============================================================
# 10. PLACEHOLDER VALUE CHECK
# ============================================================

print("\n" + "=" * 90)
print("10. PLACEHOLDER VALUE CHECK")
print("=" * 90)

placeholder_values = [
    "unknown",
    "n/a",
    "na",
    "none",
    "null",
    "-",
    ""
]

placeholder_report = []

for col in text_columns:

    normalized = (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    placeholder_count = (
        normalized.isin(
            placeholder_values
        )
        &
        df[col].notna()
    ).sum()

    if placeholder_count > 0:

        placeholder_report.append({

            "Column":
                col,

            "Placeholder_Count":
                placeholder_count
        })

placeholder_report = pd.DataFrame(
    placeholder_report
)

if not placeholder_report.empty:

    print(
        placeholder_report.to_string(
            index=False
        )
    )

else:

    print(
        "No placeholder values detected."
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("TASK 1 PROFILING COMPLETED")
print("=" * 90)

print(
    """
Checks Completed:

1. Missing values
2. Duplicate rows and movie_id
3. Detected data types
4. Malformed Release_Date values
5. Invalid Release_Year values
6. Numeric-as-text issues
7. Zero and negative values
8. Leading/trailing whitespace
9. Important category distributions
10. Placeholder values

These findings will be used to define
Data Cleaning Plan v1 in Task 2.
"""
)

TASK 1: DATA CLEANING PLAN - ISSUE IDENTIFICATION

Dataset Shape:
Rows   : 4803
Columns: 51

1. MISSING VALUES
              Column  Missing_Count  Missing_Percentage
                 ROI            869               18.09
Revenue_Budget_Ratio            869               18.09
              Profit            869               18.09
   Cast_Gender_Ratio            307                6.39
Production_Companies            123                2.56
    Third_Lead_Actor             63                1.31
   Second_Lead_Actor             53                1.10
          Lead_Actor             43                0.90
            Director             30                0.62
              Genres             10                0.21
       Release_Month              8                0.17
        Release_Year              8                0.17
              Budget              8                0.17
             Revenue              8                0.17
        Release_Date              8              

### Decide Keep / Drop / Impute Rules

In [3]:
import pandas as pd

# ==========================================
# TASK 2: DEFINE CLEANING RULES
# ==========================================

cleaning_rules = [

    {
        "Column": "movie_id",
        "Issue": "Possible duplicate identifiers",
        "Action": "DROP / REVIEW",
        "Cleaning_Rule":
            "Check duplicate movie_id values. "
            "Remove only confirmed duplicate movie records.",
        "Reason":
            "movie_id should uniquely identify each movie."
    },

    {
        "Column": "Revenue",
        "Issue": "Zero values",
        "Action": "KEEP / FLAG",
        "Cleaning_Rule":
            "Do not automatically replace zero Revenue. "
            "Retain and flag zero values unless verified as missing.",
        "Reason":
            "Replacing Revenue without evidence can distort "
            "Profit and ROI calculations."
    },

    {
        "Column": "Production_Companies",
        "Issue": "Missing values",
        "Action": "RECOVER / KEEP MISSING",
        "Cleaning_Rule":
            "Attempt to recover the actual company from reliable "
            "related data. Do not use mode imputation. "
            "If the actual company cannot be verified, retain as "
            "missing or use a documented Unknown category.",
        "Reason":
            "Assigning the most frequent company would create "
            "false company information and distort company analysis."
    },

    {
        "Column": "Primary_Production_Company",
        "Issue": "Missing or Unknown values",
        "Action": "RECOVER / STANDARDIZE",
        "Cleaning_Rule":
            "Recover from verified Production_Companies data "
            "where possible. Standardize unresolved values "
            "consistently.",
        "Reason":
            "Required for reliable company-level profitability "
            "and performance analysis."
    },

    {
        "Column": "Production_Company_Count",
        "Issue": "Zero values",
        "Action": "VALIDATE / RECALCULATE",
        "Cleaning_Rule":
            "Compare count with Production_Companies. "
            "Recalculate only when valid company information exists.",
        "Reason":
            "The count should be consistent with the actual "
            "production-company information."
    },

    {
        "Column": "Genre_Count",
        "Issue": "Zero values",
        "Action": "VALIDATE / RECALCULATE",
        "Cleaning_Rule":
            "Compare Genre_Count with Genres and recalculate "
            "when valid genre information is available.",
        "Reason":
            "Genre_Count should represent the actual number "
            "of genres assigned to the movie."
    },

    {
        "Column": "Cast_Gender_Ratio",
        "Issue": "Missing or zero values",
        "Action": "RECALCULATE / KEEP MISSING",
        "Cleaning_Rule":
            "Recalculate from available male and female cast "
            "counts where mathematically valid. "
            "Do not fabricate a ratio when required data is missing.",
        "Reason":
            "Derived values should be calculated from source "
            "columns rather than statistically guessed."
    },

    {
        "Column": "Release_Date",
        "Issue": "Potential date-format issues",
        "Action": "TYPE FIX",
        "Cleaning_Rule":
            "Convert valid values to a consistent date format. "
            "Flag malformed dates instead of guessing.",
        "Reason":
            "Consistent dates are required for release trend "
            "and seasonality analysis."
    },

    {
        "Column": "Release_Year",
        "Issue": "Potential type inconsistencies",
        "Action": "TYPE FIX / VALIDATE",
        "Cleaning_Rule":
            "Convert to numeric integer format and validate "
            "against Release_Date where available.",
        "Reason":
            "Ensures accurate year-based analysis."
    },

    {
        "Column": "Text Columns",
        "Issue": "Whitespace and inconsistent formatting",
        "Action": "HARMONIZE",
        "Cleaning_Rule":
            "Trim leading and trailing whitespace and standardize "
            "equivalent category formatting without changing "
            "legitimate names.",
        "Reason":
            "Prevents the same category from appearing as "
            "multiple values due to formatting differences."
    }

]

# Create cleaning plan dataframe
cleaning_plan = pd.DataFrame(cleaning_rules)

print("=" * 100)
print("DATA CLEANING PLAN v1")
print("=" * 100)

display(cleaning_plan) 

DATA CLEANING PLAN v1


,Column,Issue,Action,Cleaning_Rule,Reason
0,movie_id,Possible duplicate identifiers,DROP / REVIEW,Check duplicate movie_id values. Remove only c...,movie_id should uniquely identify each movie.
1,Revenue,Zero values,KEEP / FLAG,Do not automatically replace zero Revenue. Ret...,Replacing Revenue without evidence can distort...
2,Production_Companies,Missing values,RECOVER / KEEP MISSING,Attempt to recover the actual company from rel...,Assigning the most frequent company would crea...
3,Primary_Production_Company,Missing or Unknown values,RECOVER / STANDARDIZE,Recover from verified Production_Companies dat...,Required for reliable company-level profitabil...
4,Production_Company_Count,Zero values,VALIDATE / RECALCULATE,Compare count with Production_Companies. Recal...,The count should be consistent with the actual...
5,Genre_Count,Zero values,VALIDATE / RECALCULATE,Compare Genre_Count with Genres and recalculat...,Genre_Count should represent the actual number...
6,Cast_Gender_Ratio,Missing or zero values,RECALCULATE / KEEP MISSING,Recalculate from available male and female cas...,Derived values should be calculated from sourc...
7,Release_Date,Potential date-format issues,TYPE FIX,Convert valid values to a consistent date form...,Consistent dates are required for release tren...
8,Release_Year,Potential type inconsistencies,TYPE FIX / VALIDATE,Convert to numeric integer format and validate...,Ensures accurate year-based analysis.
9,Text Columns,Whitespace and inconsistent formatting,HARMONIZE,Trim leading and trailing whitespace and stand...,Prevents the same category from appearing as m...


### Freeze Cleaning Plan v1 and Save to /docs

In [4]:
import os
from datetime import datetime

# ==========================================
# TASK 3: FREEZE DATA CLEANING PLAN v1
# ==========================================

# Create docs folder if it does not exist
os.makedirs("../docs", exist_ok=True)

# Add plan metadata
cleaning_plan["Plan_Version"] = "v1"
cleaning_plan["Plan_Status"] = "FROZEN"

cleaning_plan["Created_Date"] = (
    datetime.now().strftime("%Y-%m-%d")
)

# Arrange columns
cleaning_plan = cleaning_plan[[
    "Plan_Version",
    "Plan_Status",
    "Column",
    "Issue",
    "Action",
    "Cleaning_Rule",
    "Reason",
    "Created_Date"
]]

# Save frozen plan
output_path = "../docs/Data_Cleaning_Plan_v1.csv"

cleaning_plan.to_csv(
    output_path,
    index=False
)

print("=" * 80)
print("DATA CLEANING PLAN v1 FROZEN")
print("=" * 80)

print("Version      : v1")
print("Status       : FROZEN")
print("Total Rules  :", len(cleaning_plan))
print("Saved To     :", output_path)

print(
    "\nIMPORTANT:"
    "\nThe cleaning plan has been frozen."
    "\nDo not change cleaning rules during implementation "
    "without documenting a new version."
)

DATA CLEANING PLAN v1 FROZEN
Version      : v1
Status       : FROZEN
Total Rules  : 10
Saved To     : ../docs/Data_Cleaning_Plan_v1.csv

IMPORTANT:
The cleaning plan has been frozen.
Do not change cleaning rules during implementation without documenting a new version.
